# Stage 03 - Silver Canonicalization and Quarantine

Canonicalize shared identifiers, align times to UTC, normalize units, deduplicate messages, flag late arrivals, and expose quarantine instead of silently dropping bad data.

In [ ]:
from pyspark.sql import Window, functions as F

bronze_observed_df = spark.table("bronze_observed_test_events")
bronze_operations_df = spark.table("bronze_operations_snapshot")
bronze_baseline_df = spark.table("bronze_emulated_test_baseline")

demo_late_record_threshold_seconds = 90
uuid_pattern = "^[0-9a-fA-F]{8}-[0-9a-fA-F]{4}-4[0-9a-fA-F]{3}-8[0-9a-fA-F]{3}-[0-9a-fA-F]{12}$"

def null_if_blank(column_name):
    return F.when(F.trim(F.col(column_name)) == "", F.lit(None)).otherwise(F.col(column_name))

In [ ]:
silver_baseline_df = (
    bronze_baseline_df
    .withColumn("site_id", null_if_blank("site_id"))
    .withColumn("system_instance_id", null_if_blank("system_instance_id"))
    .withColumn("track_id", null_if_blank("track_id"))
    .withColumn("planned_event_time_ts", F.to_timestamp("planned_event_time_utc"))
    .withColumn("predicted_value_double", F.col("predicted_value").cast("double"))
    .withColumn("normalized_predicted_value", F.col("predicted_value").cast("double"))
    .withColumn("normalized_predicted_unit", F.col("predicted_unit"))
    .withColumn(
        "comparison_key",
        F.when(
            F.col("planned_event_type") == "operations.snapshot",
            F.concat_ws("|", F.col("planned_event_type"), F.col("required_feed_id"), F.coalesce(F.col("system_instance_id"), F.lit("ALL"))),
        ).otherwise(
            F.concat_ws("|", F.col("planned_event_type"), F.coalesce(F.col("track_id"), F.lit("NO_TRACK")), F.coalesce(F.col("system_instance_id"), F.lit("ALL"))),
        )
    )
)

allowed_sites = [row["site_id"] for row in silver_baseline_df.select("site_id").distinct().collect() if row["site_id"]]
allowed_systems = [row["system_instance_id"] for row in silver_baseline_df.select("system_instance_id").where(F.col("system_instance_id").isNotNull()).distinct().collect()]
allowed_objectives = [row["test_objective_id"] for row in silver_baseline_df.select("test_objective_id").distinct().collect()]

print(f"Baseline rows ready for comparison: {silver_baseline_df.count()}")
print(f"Canonical sites: {allowed_sites}")
print(f"Canonical systems: {allowed_systems}")

In [ ]:
observed_staged_df = (
    bronze_observed_df
    .withColumn("site_id", null_if_blank("site_id"))
    .withColumn("source_instance_id", null_if_blank("source_instance_id"))
    .withColumn("track_id", null_if_blank("track_id"))
    .withColumn("event_time_ts", F.to_timestamp("event_time_utc"))
    .withColumn("ingest_time_ts", F.to_timestamp("ingest_time_utc"))
    .withColumn(
        "required_feed_id",
        F.when(F.col("event_type") == "sensor.observation", F.lit("feed-sensor-observation"))
        .when(F.col("event_type") == "command.integration", F.lit("feed-command-integration"))
        .when(F.col("event_type") == "system.status", F.lit("feed-system-status"))
        .otherwise(F.lit("feed-review")),
    )
    .withColumn(
        "actual_state",
        F.when(F.col("event_type") == "sensor.observation", F.col("observation_type"))
        .when(F.col("event_type") == "command.integration", F.col("status"))
        .when(F.col("event_type") == "system.status", F.col("readiness_state")),
    )
    .withColumn(
        "normalized_measure_value",
        F.when(F.col("event_type") == "sensor.observation", F.col("quality.score"))
        .when(F.col("event_type") == "command.integration", F.col("processing_delay_ms") / F.lit(1000.0))
        .when(F.col("event_type") == "system.status", F.col("health_score")),
    )
    .withColumn(
        "normalized_measure_unit",
        F.when(F.col("event_type") == "sensor.observation", F.lit("quality_fraction"))
        .when(F.col("event_type") == "command.integration", F.lit("seconds"))
        .when(F.col("event_type") == "system.status", F.lit("health_fraction")),
    )
    .withColumn(
        "comparison_key",
        F.concat_ws("|", F.col("event_type"), F.coalesce(F.col("track_id"), F.lit("NO_TRACK")), F.coalesce(F.col("source_instance_id"), F.lit("ALL")))
    )
    .withColumn("ingest_lag_seconds", F.unix_timestamp("ingest_time_ts") - F.unix_timestamp("event_time_ts"))
    .withColumn("is_late_record", F.col("ingest_lag_seconds") > F.lit(demo_late_record_threshold_seconds))
    .withColumn("uuid_is_valid", F.col("event_id").rlike(uuid_pattern))
    .withColumn("site_is_known", F.col("site_id").isin(allowed_sites))
    .withColumn("system_is_known", F.col("source_instance_id").isin(allowed_systems))
    .withColumn(
        "validation_reason",
        F.when(~F.col("uuid_is_valid"), F.lit("INVALID_EVENT_ID"))
        .when(F.col("event_time_ts").isNull(), F.lit("INVALID_EVENT_TIME"))
        .when(F.col("ingest_time_ts").isNull(), F.lit("INVALID_INGEST_TIME"))
        .when(F.col("site_id").isNull(), F.lit("MISSING_SITE_ID"))
        .when(~F.col("site_is_known"), F.lit("UNKNOWN_SITE_ID"))
        .when(F.col("source_instance_id").isNull(), F.lit("MISSING_SOURCE_INSTANCE_ID"))
        .when(~F.col("system_is_known"), F.lit("UNKNOWN_SOURCE_INSTANCE_ID"))
    )
)

observed_window = Window.partitionBy("event_id").orderBy(F.col("ingest_time_ts").desc_nulls_last(), F.col("bronze_record_sha256").desc())
observed_ranked_df = observed_staged_df.withColumn("event_rank", F.row_number().over(observed_window))

silver_observed_df = observed_ranked_df.filter(F.col("validation_reason").isNull() & (F.col("event_rank") == 1))

observed_quarantine_df = (
    observed_ranked_df
    .filter(F.col("validation_reason").isNotNull() | (F.col("event_rank") > 1))
    .withColumn("quarantine_reason", F.coalesce(F.col("validation_reason"), F.lit("DUPLICATE_EVENT_ID")))
    .select(
        F.lit("observed_events").alias("source_lane"),
        F.col("event_id").alias("record_id"),
        "quarantine_reason",
        "event_type",
        "required_feed_id",
        "site_id",
        F.col("source_instance_id").alias("system_instance_id"),
        F.col("event_time_utc").alias("raw_time_text"),
        F.to_json(F.struct(*[F.col(column_name) for column_name in observed_ranked_df.columns])).alias("raw_payload_json"),
    )
)

print(f"Valid observed records: {silver_observed_df.count()}")
print(f"Observed quarantine records: {observed_quarantine_df.count()}")

In [ ]:
operations_staged_df = (
    bronze_operations_df
    .withColumn("site_id", null_if_blank("site_id"))
    .withColumn("system_instance_id", null_if_blank("system_instance_id"))
    .withColumn("reported_time_local_ts", F.to_timestamp("reported_time_local", "yyyy-MM-dd HH:mm:ss"))
    .withColumn(
        "reported_time_utc",
        F.to_timestamp(F.from_unixtime(F.unix_timestamp("reported_time_local", "yyyy-MM-dd HH:mm:ss") - F.col("utc_offset_minutes").cast("int") * 60))
    )
    .withColumn("measure_value_double", F.col("measure_value").cast("double"))
    .withColumn(
        "normalized_measure_value",
        F.when(F.col("measure_unit") == "minutes", F.col("measure_value_double") * F.lit(60.0)).otherwise(F.col("measure_value_double"))
    )
    .withColumn(
        "normalized_measure_unit",
        F.when(F.col("measure_unit") == "minutes", F.lit("seconds")).otherwise(F.col("measure_unit"))
    )
    .withColumn(
        "comparison_key",
        F.when(
            F.col("record_type") == "FEED_HEALTH",
            F.concat_ws("|", F.lit("operations.snapshot"), F.col("required_feed_id"), F.coalesce(F.col("system_instance_id"), F.lit("ALL")))
        ).otherwise(
            F.concat_ws("|", F.col("record_type"), F.coalesce(F.col("system_instance_id"), F.lit("ALL")), F.col("required_feed_id"))
        )
    )
    .withColumn(
        "validation_reason",
        F.when(F.col("reported_time_local_ts").isNull(), F.lit("INVALID_REPORTED_TIME"))
        .when(F.col("reported_time_utc").isNull(), F.lit("INVALID_TIME_REFERENCE"))
        .when(~F.col("test_objective_id").isin(allowed_objectives), F.lit("UNKNOWN_TEST_OBJECTIVE"))
    )
)

silver_operations_df = operations_staged_df.filter(F.col("validation_reason").isNull())

operations_quarantine_df = (
    operations_staged_df
    .filter(F.col("validation_reason").isNotNull())
    .select(
        F.lit("operations_snapshot").alias("source_lane"),
        F.col("snapshot_record_id").alias("record_id"),
        F.col("validation_reason").alias("quarantine_reason"),
        F.col("record_type").alias("event_type"),
        "required_feed_id",
        "site_id",
        "system_instance_id",
        F.col("reported_time_local").alias("raw_time_text"),
        F.to_json(F.struct(*[F.col(column_name) for column_name in operations_staged_df.columns])).alias("raw_payload_json"),
    )
)

observed_feed_counts_df = silver_observed_df.groupBy("required_feed_id").agg(
    F.count("*").alias("valid_observed_count"),
    F.sum(F.when(F.col("is_late_record"), 1).otherwise(0)).alias("late_record_count"),
)

feed_health_df = silver_operations_df.filter(F.col("record_type") == "FEED_HEALTH").select(
    "scenario_id",
    "test_event_id",
    "test_objective_id",
    "required_feed_id",
    "system_instance_id",
    "reported_state",
    "expected_state",
    F.col("normalized_measure_value").alias("snapshot_record_count"),
    "notes",
)

silver_required_feed_checks_df = (
    silver_baseline_df.select("scenario_id", "test_event_id", "test_objective_id", "required_feed_id").distinct()
    .join(feed_health_df, ["scenario_id", "test_event_id", "test_objective_id", "required_feed_id"], "left")
    .join(observed_feed_counts_df, ["required_feed_id"], "left")
    .fillna({"valid_observed_count": 0, "late_record_count": 0, "snapshot_record_count": 0})
    .withColumn(
        "required_feed_status",
        F.when(F.col("reported_state") == "MISSING", F.lit("MISSING"))
        .when(F.col("reported_state") == "AVAILABLE", F.lit("AVAILABLE"))
        .otherwise(F.lit("REVIEW"))
    )
    .withColumn(
        "setup_check_result",
        F.when(F.col("required_feed_status") == "MISSING", F.lit("FAIL"))
        .when((F.col("valid_observed_count") == 0) & (F.col("required_feed_id") != "feed-sustainment"), F.lit("FAIL"))
        .otherwise(F.lit("PASS"))
    )
)

print(f"Canonical operations records: {silver_operations_df.count()}")
print(f"Required feed checks: {silver_required_feed_checks_df.count()}")

In [ ]:
silver_quarantine_df = (
    observed_quarantine_df.unionByName(operations_quarantine_df, allowMissingColumns=True)
    .withColumn(
        "source_name",
        F.when(F.col("source_lane") == "observed", F.lit("Observed test events"))
        .when(F.col("source_lane") == "operations", F.lit("Operations readiness snapshot"))
        .otherwise(F.initcap(F.regexp_replace(F.col("source_lane"), "_", " "))),
    )
    .withColumn(
        "quality_issue",
        F.when(F.col("quarantine_reason") == "INVALID_EVENT_TIME", F.lit("Event time could not be validated"))
        .when(F.col("quarantine_reason") == "DUPLICATE_EVENT_ID", F.lit("Duplicate event was isolated"))
        .when(F.col("quarantine_reason") == "INVALID_REPORTED_TIME", F.lit("Reported time could not be validated"))
        .otherwise(F.initcap(F.regexp_replace(F.col("quarantine_reason"), "_", " "))),
    )
)

silver_tables = {
    "silver_emulated_test_baseline": silver_baseline_df,
    "silver_observed_events": silver_observed_df,
    "silver_operations_snapshot": silver_operations_df,
    "silver_required_feed_checks": silver_required_feed_checks_df,
    "silver_quarantine_records": silver_quarantine_df,
}

for table_name, frame in silver_tables.items():
    (
        frame.write.format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(table_name)
    )

print("Silver tables are ready for Gold product assembly.")

In [ ]:
spark.table("silver_observed_events").select(
    "event_id",
    "event_type",
    "required_feed_id",
    "source_instance_id",
    "actual_state",
    "normalized_measure_value",
    "normalized_measure_unit",
    "is_late_record",
).orderBy("event_type", "event_id").show(truncate=False)

spark.table("silver_required_feed_checks").orderBy("required_feed_id").show(truncate=False)
spark.table("silver_quarantine_records").orderBy("source_lane", "record_id").show(truncate=False)